In [1]:
# THIS REPORT BUILDER WAS ADAPTED FROM ANTHROPIC LEARNING CONTENT 2025
# Report Builder for prompt/model WER evaluation results
#
# Input: a dict built from prompt-eval output
#   {
#       "set1": {
#           "prompt": "Defence",
#           "model1tested": "whisper-1", "wer_score1": 0.0899,
#           "model2tested": "whisper-large-v3", "wer_score2": 0.0262,
#           "model3tested": "whisper-large-v3-turbo", "wer_score3": 0.2321,
#       },
#       "set2": {...},
#       ...
#   }
# The keys inside each set don't need to follow a strict schema - numbered
# "model1tested"/"wer_score1" style keys, unnumbered "model"/"wer_score" pairs
# in encounter order, or a plain {"scores": {model: wer}} shape all work. A
# list of sets is accepted too. Any number of sets can be passed.

import html as _html

# $/hour transcription pricing, used to plot WER vs. cost per configuration
MODEL_PRICING_PER_HOUR = {
    "whisper-1": 3.6,
    "whisper-large-v3": 0.111,
    "whisper-large-v3-turbo": 0.04,
}


def _normalize_set(raw_set):
    """Pull a prompt string and an ordered list of (model_name, wer_score) pairs
    out of a loosely-structured set dict."""
    prompt_text = None
    pairs = []
    pending_model = None

    if "scores" in raw_set and isinstance(raw_set["scores"], dict):
        prompt_text = raw_set.get("prompt", "")
        pairs = list(raw_set["scores"].items())
        return prompt_text, pairs

    model_items = []
    score_items = []
    for key, value in raw_set.items():
        key_lower = str(key).lower()
        if "prompt" in key_lower:
            prompt_text = value
        elif "model" in key_lower:
            model_items.append(value)
        elif "wer" in key_lower or "score" in key_lower:
            score_items.append(value)

    if model_items and score_items:
        # numbered keys (wer_score1/wer_score2/...) collect here 1:1; a
        # repeated plain key like "wer_score" collapses in a real dict, so
        # this just zips whatever counts do line up
        for i in range(min(len(model_items), len(score_items))):
            pairs.append((model_items[i], score_items[i]))
    else:
        # interleaved model/score keys with no numbering - pair by encounter order
        for key, value in raw_set.items():
            key_lower = str(key).lower()
            if "model" in key_lower:
                pending_model = value
            elif "wer" in key_lower or "score" in key_lower:
                model_name = pending_model if pending_model is not None else f"model{len(pairs) + 1}"
                pairs.append((model_name, value))
                pending_model = None

    return prompt_text or "", pairs


def _compute_pareto_frontier(points):
    """points: list of dicts with 'price' and 'wer'. Lower is better on both
    axes, so a point is on the frontier if no other point dominates it
    (equal-or-lower price AND equal-or-lower WER, with at least one strictly lower)."""
    frontier = []
    for p in points:
        dominated = False
        for q in points:
            if q is p:
                continue
            if (
                q["price"] <= p["price"]
                and q["wer"] <= p["wer"]
                and (q["price"] < p["price"] or q["wer"] < p["wer"])
            ):
                dominated = True
                break
        if not dominated:
            frontier.append(p)
    frontier.sort(key=lambda p: p["price"])
    return frontier


_PALETTE = ["#5c6bc0", "#26a69a", "#ef6c00", "#ab47bc", "#8d6e63", "#42a5f5", "#d4527b"]

# marker shapes cycled across prompts, in order of first appearance
_SHAPES = ["circle", "square", "triangle", "diamond", "star", "cross", "pentagon"]


def _truncate_label(text, max_len=28):
    text = str(text)
    return text if len(text) <= max_len else text[: max_len - 1] + "…"


def _shape_markup(shape, cx, cy, size, fill, stroke_attr):
    """Return SVG markup for a single marker of the given shape, centered at (cx, cy)."""
    if shape == "square":
        s = size * 1.6
        return f'<rect x="{cx - s / 2:.1f}" y="{cy - s / 2:.1f}" width="{s:.1f}" height="{s:.1f}" fill="{fill}" {stroke_attr} />'
    if shape == "triangle":
        s = size * 1.8
        h = s * 0.87
        pts = f"{cx:.1f},{cy - h * 2 / 3:.1f} {cx - s / 2:.1f},{cy + h / 3:.1f} {cx + s / 2:.1f},{cy + h / 3:.1f}"
        return f'<polygon points="{pts}" fill="{fill}" {stroke_attr} />'
    if shape == "diamond":
        s = size * 1.3
        pts = f"{cx:.1f},{cy - s:.1f} {cx + s:.1f},{cy:.1f} {cx:.1f},{cy + s:.1f} {cx - s:.1f},{cy:.1f}"
        return f'<polygon points="{pts}" fill="{fill}" {stroke_attr} />'
    if shape == "star":
        import math
        outer, inner = size * 1.3, size * 0.55
        pts = []
        for i in range(10):
            r = outer if i % 2 == 0 else inner
            angle = math.pi / 2 + i * math.pi / 5
            pts.append(f"{cx + r * math.cos(angle):.1f},{cy - r * math.sin(angle):.1f}")
        return f'<polygon points="{" ".join(pts)}" fill="{fill}" {stroke_attr} />'
    if shape == "cross":
        arm = size * 0.55
        thick = size * 0.55
        pts = (
            f"{cx - thick:.1f},{cy - arm - thick:.1f} {cx + thick:.1f},{cy - arm - thick:.1f} "
            f"{cx + thick:.1f},{cy - thick:.1f} {cx + arm + thick:.1f},{cy - thick:.1f} "
            f"{cx + arm + thick:.1f},{cy + thick:.1f} {cx + thick:.1f},{cy + thick:.1f} "
            f"{cx + thick:.1f},{cy + arm + thick:.1f} {cx - thick:.1f},{cy + arm + thick:.1f} "
            f"{cx - thick:.1f},{cy + thick:.1f} {cx - arm - thick:.1f},{cy + thick:.1f} "
            f"{cx - arm - thick:.1f},{cy - thick:.1f} {cx - thick:.1f},{cy - thick:.1f}"
        )
        return f'<polygon points="{pts}" fill="{fill}" {stroke_attr} />'
    if shape == "pentagon":
        import math
        r = size * 1.3
        pts = []
        for i in range(5):
            angle = math.pi / 2 + i * 2 * math.pi / 5
            pts.append(f"{cx + r * math.cos(angle):.1f},{cy - r * math.sin(angle):.1f}")
        return f'<polygon points="{" ".join(pts)}" fill="{fill}" {stroke_attr} />'
    # default: circle
    return f'<circle cx="{cx:.1f}" cy="{cy:.1f}" r="{size:.1f}" fill="{fill}" {stroke_attr} />'


def _build_pareto_chart_svg(points, pricing):
    if not points:
        return "<p>No configurations with known pricing to plot.</p>"

    width, height = 720, 460
    margin = {"top": 20, "right": 30, "bottom": 60, "left": 70}
    plot_w = width - margin["left"] - margin["right"]
    plot_h = height - margin["top"] - margin["bottom"]

    prices = [p["price"] for p in points]
    wers = [p["wer"] for p in points]
    price_min, price_max = min(prices), max(prices)
    wer_min, wer_max = min(wers), max(wers)

    # pad ranges so points don't sit on the axes
    price_pad = (price_max - price_min) * 0.1 or max(price_max, 1) * 0.1
    wer_pad = (wer_max - wer_min) * 0.1 or 0.01
    price_lo, price_hi = max(0, price_min - price_pad), price_max + price_pad
    wer_lo, wer_hi = max(0, wer_min - wer_pad), wer_max + wer_pad

    def x_of(price):
        # reversed so lower price (better) plots further right
        return margin["left"] + plot_w - (price - price_lo) / (price_hi - price_lo) * plot_w

    def y_of(wer):
        # reversed so lower WER (better) plots higher up the chart
        return margin["top"] + (wer - wer_lo) / (wer_hi - wer_lo) * plot_h

    prompt_labels = []
    for p in points:
        if p["prompt_label"] not in prompt_labels:
            prompt_labels.append(p["prompt_label"])
    shape_by_prompt = {label: _SHAPES[i % len(_SHAPES)] for i, label in enumerate(prompt_labels)}

    model_names = []
    for p in points:
        if p["model"] not in model_names:
            model_names.append(p["model"])
    color_by_model = {name: _PALETTE[i % len(_PALETTE)] for i, name in enumerate(model_names)}

    frontier = _compute_pareto_frontier(points)
    frontier_line = " ".join(f"{x_of(p['price']):.1f},{y_of(p['wer']):.1f}" for p in frontier)

    # gridlines + axis ticks (5 steps each)
    grid_svg = ""
    x_ticks = ""
    for i in range(6):
        gx = margin["left"] + plot_w * i / 5
        # left edge (i=0) is the highest (most expensive) price, since the axis is reversed
        val = price_hi - (price_hi - price_lo) * i / 5
        grid_svg += f'<line x1="{gx:.1f}" y1="{margin["top"]}" x2="{gx:.1f}" y2="{margin["top"] + plot_h}" stroke="#eee" />'
        x_ticks += f'<text x="{gx:.1f}" y="{margin["top"] + plot_h + 18}" font-size="11" text-anchor="middle" fill="#555">${val:.2f}</text>'
    y_ticks = ""
    for i in range(6):
        gy = margin["top"] + plot_h * i / 5
        # top of the chart (i=0) is the lowest (best) WER value, since the axis is reversed
        val = wer_lo + (wer_hi - wer_lo) * i / 5
        grid_svg += f'<line x1="{margin["left"]}" y1="{gy:.1f}" x2="{margin["left"] + plot_w}" y2="{gy:.1f}" stroke="#eee" />'
        y_ticks += f'<text x="{margin["left"] - 10}" y="{gy + 4:.1f}" font-size="11" text-anchor="end" fill="#555">{val:.3f}</text>'

    points_svg = ""
    for p in points:
        cx, cy = x_of(p["price"]), y_of(p["wer"])
        color = color_by_model[p["model"]]
        shape = shape_by_prompt[p["prompt_label"]]
        on_frontier = p in frontier
        size = 8 if on_frontier else 6
        stroke_attr = 'stroke="#222" stroke-width="2"' if on_frontier else 'stroke="#fff" stroke-width="1.5"'
        tooltip_label = _html.escape(f"{p['model']} / {p['prompt_label']}")
        marker = _shape_markup(shape, cx, cy, size, color, stroke_attr)
        points_svg += f"""
            <g>
                {marker}
                <title>{tooltip_label}&#10;${p['price']:.3f}/hr, WER {p['wer']:.4f}</title>
            </g>
        """

    # legend: model colors on the left, prompt shapes on the right, stacked in their own columns
    legend_svg = ""
    legend_col_x = width - margin["right"] - 300
    for i, name in enumerate(model_names):
        ly = margin["top"] + i * 18
        legend_svg += f"""
            <circle cx="{legend_col_x:.1f}" cy="{ly + 4}" r="5" fill="{color_by_model[name]}" stroke="#222" stroke-width="1" />
            <text x="{legend_col_x + 12:.1f}" y="{ly + 8}" font-size="11" fill="#333">{_html.escape(_truncate_label(name, 24))}</text>
        """
    shape_col_x = width - margin["right"] - 130
    for i, label in enumerate(prompt_labels):
        ly = margin["top"] + i * 18
        marker = _shape_markup(shape_by_prompt[label], shape_col_x, ly + 4, 6, "#666", 'stroke="#222" stroke-width="1"')
        legend_svg += f"""
            {marker}
            <text x="{shape_col_x + 12:.1f}" y="{ly + 8}" font-size="11" fill="#333">{_html.escape(_truncate_label(label, 28))}</text>
        """

    return f"""
    <svg viewBox="0 0 {width} {height}" width="100%" style="max-width: {width}px; font-family: Arial, sans-serif;">
        {grid_svg}
        <line x1="{margin['left']}" y1="{margin['top']}" x2="{margin['left']}" y2="{margin['top'] + plot_h}" stroke="#999" />
        <line x1="{margin['left']}" y1="{margin['top'] + plot_h}" x2="{margin['left'] + plot_w}" y2="{margin['top'] + plot_h}" stroke="#999" />
        {x_ticks}
        {y_ticks}
        <text x="{margin['left'] + plot_w / 2}" y="{height - 10}" font-size="12" text-anchor="middle" fill="#333">Price ($ / hour)</text>
        <text x="14" y="{margin['top'] + plot_h / 2}" font-size="12" text-anchor="middle" fill="#333" transform="rotate(-90 14 {margin['top'] + plot_h / 2})">WER Score (lower = better accuracy)</text>
        <polyline points="{frontier_line}" fill="none" stroke="#222" stroke-width="1.5" stroke-dasharray="4 3" />
        {points_svg}
        {legend_svg}
    </svg>
    """


def _build_pareto_section(normalized, pricing):
    # datapoints are strictly (model, prompt) pairs pulled straight from the
    # per-prompt scores - no cross-prompt averages are ever added here
    points = []
    for prompt_text, pairs in normalized:
        prompt_label = str(prompt_text)
        for model_name, wer in pairs:
            price = pricing.get(model_name)
            if price is None:
                continue
            points.append({
                "model": model_name,
                "prompt_label": prompt_label,
                "price": price,
                "wer": wer,
            })

    chart_svg = _build_pareto_chart_svg(points, pricing)

    return f"""
        <div class="prompt-section">
            <h2>Pareto Frontier: Price vs. WER</h2>
            <p style="color:#555; font-size: 13px;">Each point is one model+prompt configuration (color = model, shape = prompt). The dashed line connects configurations where no other configuration is both cheaper and more accurate. Both axes are oriented so the better direction (lower price, lower WER) points toward the top-right.</p>
            {chart_svg}
        </div>
    """


def generate_prompt_evaluation_report(prompt_sets, pricing=None):
    if isinstance(prompt_sets, dict):
        raw_sets = list(prompt_sets.values())
    else:
        raw_sets = list(prompt_sets)

    normalized = [_normalize_set(raw_set) for raw_set in raw_sets]
    pricing = pricing if pricing is not None else MODEL_PRICING_PER_HOUR

    total_prompts = len(normalized)
    all_models = []
    for _, pairs in normalized:
        for model_name, _score in pairs:
            if model_name not in all_models:
                all_models.append(model_name)

    best_overall = None
    for prompt_text, pairs in normalized:
        for model_name, score in pairs:
            if best_overall is None or score < best_overall[2]:
                best_overall = (prompt_text, model_name, score)

    sections_html = ""
    for prompt_text, pairs in normalized:
        scores = dict(pairs)
        max_score = max(scores.values()) if scores else 0
        best_model = min(scores, key=scores.get) if scores else None

        rows_html = ""
        for model_name, score in pairs:
            bar_pct = (score / max_score * 100) if max_score > 0 else 0
            is_best = model_name == best_model
            rows_html += f"""
                <div class="bar-row">
                    <div class="bar-label">{_html.escape(str(model_name))}{' <span class="best-tag">best</span>' if is_best else ''}</div>
                    <div class="bar-track">
                        <div class="bar-fill{' bar-fill-best' if is_best else ''}" style="width: {bar_pct:.2f}%;"></div>
                    </div>
                    <div class="bar-value">{score:.4f}</div>
                </div>
            """

        sections_html += f"""
            <div class="prompt-section">
                <h2>Prompt: <span class="prompt-text">&ldquo;{_html.escape(str(prompt_text))}&rdquo;</span></h2>
                <div class="bars">
                    {rows_html}
                </div>
            </div>
        """

    pareto_section_html = _build_pareto_section(normalized, pricing)

    html_doc = f"""
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>Prompt Evaluation Report</title>
        <style>
            body {{
                font-family: Arial, sans-serif;
                line-height: 1.6;
                margin: 0;
                padding: 20px;
                color: #333;
            }}
            .header {{
                background-color: #f0f0f0;
                padding: 20px;
                border-radius: 5px;
                margin-bottom: 20px;
            }}
            .summary-stats {{
                display: flex;
                justify-content: space-between;
                flex-wrap: wrap;
                gap: 10px;
            }}
            .stat-box {{
                background-color: #fff;
                border-radius: 5px;
                padding: 15px;
                box-shadow: 0 2px 5px rgba(0,0,0,0.1);
                flex-basis: 30%;
                min-width: 200px;
            }}
            .stat-value {{
                font-size: 20px;
                font-weight: bold;
                margin-top: 5px;
            }}
            .prompt-section {{
                background-color: #fff;
                border-radius: 5px;
                padding: 20px;
                margin-bottom: 20px;
                box-shadow: 0 2px 5px rgba(0,0,0,0.1);
                overflow-x: auto;
            }}
            .prompt-section h2 {{
                margin-top: 0;
                font-size: 16px;
            }}
            .prompt-text {{
                font-weight: normal;
                font-style: italic;
                color: #555;
            }}
            .bars {{
                display: flex;
                flex-direction: column;
                gap: 10px;
                margin-top: 15px;
            }}
            .bar-row {{
                display: grid;
                grid-template-columns: 220px 1fr 90px;
                align-items: center;
                gap: 10px;
            }}
            .bar-label {{
                font-size: 14px;
                white-space: nowrap;
                overflow: hidden;
                text-overflow: ellipsis;
            }}
            .best-tag {{
                background-color: #c8e6c9;
                color: #2e7d32;
                font-size: 11px;
                font-weight: bold;
                padding: 1px 6px;
                border-radius: 3px;
            }}
            .bar-track {{
                background-color: #eee;
                border-radius: 3px;
                height: 18px;
                overflow: hidden;
            }}
            .bar-fill {{
                background-color: #ef9a9a;
                height: 100%;
            }}
            .bar-fill-best {{
                background-color: #81c784;
            }}
            .bar-value {{
                font-size: 13px;
                font-weight: bold;
                text-align: right;
            }}
            table {{
                width: 100%;
                border-collapse: collapse;
                margin-top: 20px;
            }}
            th {{
                background-color: #4a4a4a;
                color: white;
                text-align: left;
                padding: 12px;
            }}
            td {{
                padding: 10px;
                border-bottom: 1px solid #ddd;
                vertical-align: top;
            }}
            tr:nth-child(even) {{
                background-color: #f9f9f9;
            }}
        </style>
    </head>
    <body>
        <div class="header">
            <h1>Prompt Evaluation Report</h1>
            <div class="summary-stats">
                <div class="stat-box">
                    <div>Prompts Tested</div>
                    <div class="stat-value">{total_prompts}</div>
                </div>
                <div class="stat-box">
                    <div>Models Tested</div>
                    <div class="stat-value">{", ".join(str(m) for m in all_models) if all_models else "-"}</div>
                </div>
                <div class="stat-box">
                    <div>Best Overall (lowest WER)</div>
                    <div class="stat-value">{f"{best_overall[1]} @ {best_overall[2]:.4f}" if best_overall else "-"}</div>
                </div>
            </div>
        </div>

        {sections_html}

        <table>
            <thead>
                <tr>
                    <th>Prompt</th>
                    {"".join(f"<th>{_html.escape(str(m))}</th>" for m in all_models)}
                </tr>
            </thead>
            <tbody>
                {"".join(
                    "<tr><td>" + _html.escape(str(prompt_text)) + "</td>" +
                    "".join(
                        f"<td>{dict(pairs)[m]:.4f}</td>" if m in dict(pairs) else "<td>-</td>"
                        for m in all_models
                    ) + "</tr>"
                    for prompt_text, pairs in normalized
                )}
            </tbody>
        </table>

        {pareto_section_html}
    </body>
    </html>
    """

    return html_doc
